# Tutorial: Clasificaci?n Employee Churn Model

## Audiencia
- Alumnos de posgrado que desean practicar clasificaci?n supervisada con datos de recursos humanos.

## Prerrequisitos
- Conocimientos b?sicos de Python y pandas.
- Familiaridad general con entrenamiento y evaluaci?n de modelos de clasificaci?n.

## Objetivo
Construir un modelo base de **abandono de empleado** usando `scikit-learn`, evaluar su desempe?o e interpretar qu? variables se relacionan con la salida del personal.


## Ruta del ejercicio
1. Verificar dependencias.
2. Cargar el dataset localmente o desde GitHub cuando se abra en Colab.
3. Explorar la variable objetivo.
4. Preparar variables num?ricas y categ?ricas.
5. Entrenar un modelo de clasificaci?n con `LogisticRegression`.
6. Evaluar el modelo con m?tricas b?sicas.
7. Revisar una interpretaci?n inicial de coeficientes.


In [ ]:
import sys

if 'google.colab' in sys.modules:
    try:
        import sklearn  # noqa: F401
    except ImportError:
        %pip install -q scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [ ]:
RAW_DATA_URL = 'https://raw.githubusercontent.com/JuanBaldemarG/portafoliocolabJBGV/main/data/employee-churn/HR_dataset_copy.csv'
LOCAL_CANDIDATES = [
    Path('../data/employee-churn/HR_dataset_copy.csv'),
    Path('data/employee-churn/HR_dataset_copy.csv'),
    Path('/content/HR_dataset_copy.csv')
]

def load_dataset() -> pd.DataFrame:
    if 'google.colab' in sys.modules:
        return pd.read_csv(RAW_DATA_URL)

    for candidate in LOCAL_CANDIDATES:
        if candidate.exists():
            return pd.read_csv(candidate)

    return pd.read_csv(RAW_DATA_URL)

df = load_dataset()
df.head()


In [ ]:
print(f'Registros: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
display(df.dtypes.to_frame('tipo'))


In [ ]:
target = 'left'
df[target].value_counts(normalize=True).rename('proporcion').to_frame()


## Preparaci?n del modelo
Usaremos la variable `left` como objetivo. Las columnas num?ricas y categ?ricas se transforman dentro de un `Pipeline` para dejar el flujo reproducible y compatible con Colab.


In [ ]:
X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


In [ ]:
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print(f'Accuracy: {accuracy_score(y_test, pred):.4f}')
print(f'ROC AUC: {roc_auc_score(y_test, proba):.4f}')
print()
print(classification_report(y_test, pred))


In [ ]:
cm = confusion_matrix(y_test, pred)
cm_df = pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1'])
cm_df


In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]
coef_df = (
    pd.DataFrame({'feature': feature_names, 'coef': coefficients})
    .assign(abs_coef=lambda d: d['coef'].abs())
    .sort_values('abs_coef', ascending=False)
)
coef_df[['feature', 'coef']].head(10)


## Interpretaci?n r?pida
- Los coeficientes positivos empujan la predicci?n hacia abandono (`left = 1`).
- Los coeficientes negativos empujan la predicci?n hacia permanencia (`left = 0`).
- Este modelo es una l?nea base ?til para discutir desempe?o, sesgo de clase y variables explicativas.


## Ejercicio para el alumno
Pruebe una de estas extensiones:
1. Cambiar `LogisticRegression` por `RandomForestClassifier`.
2. Ajustar el umbral de clasificaci?n usando `predict_proba`.
3. Comparar resultados quitando la columna categ?rica `functional area`.


In [ ]:
# Respuesta sugerida: use este espacio para probar un segundo modelo.
# Ejemplo:
# from sklearn.ensemble import RandomForestClassifier
# ...


## Errores comunes y extensi?n
**Error com?n:** olvidar el tratamiento de variables categ?ricas y pasar texto crudo al modelo.

**Extensi?n sugerida:** agregar validaci?n cruzada y comparar varios clasificadores para decidir cu?l conviene presentar como modelo final.
